# Week 4 exercises: Exploratory Data Analysis and Preprocessing

*Complete all exercises in this notebook during week 4. The exercises are not submitted — this is purely for your own learning and for preparing for the assignments, project and exam.*

Note! You should always complete the tasks first without using AI. If you need AI to get through these tasks, then you will not be able to pass the oral exam. AI can be used to check if your code could be simplified or improved after you solve the task.

---

## Part 1: Loading and first impressions

### Exercise 1: Loading the data

The file `data/dirty_cafe_sales.csv` contains 10,000 rows of synthetic sales transactions for a cafe. The data is intentionally "dirty", containing missing values, placeholder strings such as `"ERROR"` and `"UNKNOWN"`, and columns stored as the wrong data type. The original description of the dataset can be found [here](https://www.kaggle.com/datasets/ahmedmohamed2003/cafe-sales-dirty-data-for-cleaning-training?select=dirty_cafe_sales.csv), and a copy of it can be found within the `data`  folder.

a) Load the file into a pandas DataFrame. Print the shape and the first 10 rows.

b) Use `info()` to inspect the data types. Are there columns that should be numeric but are not? Which ones and why?

c) Use `describe()` and note what happens. Why does `describe()` not show the usual numeric statistics for all columns?

In [1]:
#a)
import pandas as pd


In [3]:
df = pd.read_csv('dirty_cafe_sales.csv')
df.head(10)

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
5,TXN_2602893,Smoothie,5,4.0,20.0,Credit Card,NaN,2023-03-31
6,TXN_4433211,UNKNOWN,3,3.0,9.0,ERROR,Takeaway,2023-10-06
7,TXN_6699534,Sandwich,4,4.0,16.0,Cash,UNKNOWN,2023-10-28
8,TXN_4717867,NaN,5,3.0,15.0,NaN,Takeaway,2023-07-28
9,TXN_2064365,Sandwich,5,4.0,20.0,NaN,In-store,2023-12-31


In [4]:
df.info() #the object dtype is bad means eveything is like a string

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


In [5]:
df.describe()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
count,10000,9667,9862,9821,9827,7421,6735,9841
unique,10000,10,7,8,19,5,4,367
top,TXN_1961373,Juice,5,3.0,6.0,Digital Wallet,Takeaway,UNKNOWN
freq,1,1171,2013,2429,979,2291,3022,159


## Part 2: Understanding the data

### Exercise 2: Counting problems

Before cleaning, it is important to understand the extent and types of issues in the data.

a) Count the number of `NaN` values in each column using `isna().sum()`.

b) Several columns contain the strings `"ERROR"` and `"UNKNOWN"` instead of actual values. For each column, count how many times `"ERROR"` and `"UNKNOWN"` appear. **Hint:** you can compare a column to a string and use `.sum()`, e.g. `(df["Item"] == "ERROR").sum()`.

c) Which column has the most missing or problematic values in total (NaN + ERROR + UNKNOWN combined)?

In [7]:
#a)
df.isna().sum()

#b) 
summary = pd.DataFrame({
    "ERROR":   [(df[col] == "ERROR").sum() for col in df.columns],
    "UNKNOWN": [(df[col] == "UNKNOWN").sum() for col in df.columns],
    "NaN":     [df[col].isna().sum() for col in df.columns],
}, index=df.columns)

summary["Total"] = summary.sum(axis=1)
summary #there are simpler way btw

,ERROR,UNKNOWN,NaN,Total
Transaction ID,0,0,0,0
Item,292,344,333,969
Quantity,170,171,138,479
Price Per Unit,190,164,179,533
Total Spent,164,165,173,502
Payment Method,306,293,2579,3178
Location,358,338,3265,3961
Transaction Date,142,159,159,460


### Exercise 3: Inspecting categorical columns

a) Use `value_counts()` to check the unique values and their frequencies for the columns `Item`, `Payment Method`, and `Location`.

b) How many valid (real) product types are there once you exclude `NaN`, `"ERROR"`, and `"UNKNOWN"`? List them.

c) How many valid payment methods and location types are there? List them.

In [8]:
print(df.value_counts(['Item']))
print()
print(df.value_counts(['Payment Method']))
print()
print(df.value_counts(['Location']))

Item    
Juice       1171
Coffee      1165
Salad       1148
Cake        1139
Sandwich    1131
Smoothie    1096
Cookie      1092
Tea         1089
UNKNOWN      344
ERROR        292
Name: count, dtype: int64

Payment Method
Digital Wallet    2291
Credit Card       2273
Cash              2258
ERROR              306
UNKNOWN            293
Name: count, dtype: int64

Location
Takeaway    3022
In-store    3017
ERROR        358
UNKNOWN      338
Name: count, dtype: int64


In [13]:
#b)
valid_items = df['Item'].dropna()
valid_items = valid_items[~valid_items.isin(['ERROR', 'UNKNOWN'])]
print(valid_items.nunique())

8


In [16]:
valid_payment = df['Payment Method'].dropna()
valid_payment = valid_payment[~valid_payment.isin(['ERROR', 'UNKNOWN'])]
print(valid_payment.unique())
print(valid_payment.nunique())
#same for location

['Credit Card' 'Cash' 'Digital Wallet']
3


## Part 3: Cleaning the data

### Exercise 4: Replacing placeholders with NaN

The strings `"ERROR"` and `"UNKNOWN"` are not real data — they indicate that the value is missing or invalid. Treating them uniformly as `NaN` makes further cleaning much simpler.

a) Replace all occurrences of `"ERROR"` and `"UNKNOWN"` in the entire DataFrame with `np.nan`. **Hint:** `df.replace()` can accept a list of values to replace.

b) Verify the replacement by checking `isna().sum()` again. The counts should now be higher than in Exercise 2a because the ERROR/UNKNOWN values have been converted to NaN.

c) What percentage of values is missing in each column? Compute this and print it rounded to one decimal. **Hint:** divide the NaN count by the total number of rows and multiply by 100.

In [18]:
import numpy as np

df = df.replace(['ERROR', 'UNKNOWN'], np.nan)
df



,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,NaN,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,NaN,NaN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2,2.0,4.0,NaN,NaN,2023-08-30
9996,TXN_9659401,NaN,3,NaN,3.0,Digital Wallet,NaN,2023-06-02
9997,TXN_5255387,Coffee,4,2.0,8.0,Digital Wallet,NaN,2023-03-02
9998,TXN_7695629,Cookie,3,NaN,3.0,Digital Wallet,NaN,2023-12-02


In [19]:
df.isna().sum()

Transaction ID         0
Item                 969
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64

In [ ]:
#c)

### Exercise 5: Converting data types

After replacing the placeholder strings, the columns `Quantity`, `Price Per Unit`, and `Total Spent` still have the `object` (string) data type even though they should be numeric.

a) Convert these three columns to numeric using `pd.to_numeric()`. Use the parameter `errors="coerce"` so that any remaining non-numeric strings become `NaN` rather than causing an error.

b) Convert `Transaction Date` to datetime using `pd.to_datetime()`, again with `errors="coerce"`.

c) Verify the conversions by printing `df.dtypes`.

In [ ]:
# YOUR CODE HERE


### Exercise 6: Deciding what to drop

Now we need to decide how to handle the missing values. There is no single correct answer and the right approach depends on the analysis goals.

a) Look at the missing-value percentages you computed in Exercise 4c. Are there any columns where more than 25% of the data is missing? If so, consider whether those columns are essential for analysis. Drop any columns that you judge to be too incomplete to be useful. Write a comment justifying your decision.

b) Drop all rows that still have `NaN` in the `Item` column (we cannot analyse a sale if we do not know what was sold).

c) Print the shape of the DataFrame after these steps. How many rows remain?

In [ ]:
# YOUR CODE HERE


## Part 4: Exploratory Data Analysis

### Exercise 7: Summary statistics

a) Use `describe()` on the cleaned DataFrame. Now that the columns have the correct types, you should see proper numeric summaries. Are there any values that look suspicious (e.g. negative quantities, unreasonably high prices)?

b) Use `value_counts()` on the `Item` column. Which product is the most frequently sold? Which is the least?

c) Compute the mean `Total Spent` per item using `groupby()` and `mean()`. Which item generates the highest average transaction value?

In [ ]:
# YOUR CODE HERE


### Exercise 8: Visualizing distributions

a) Import `matplotlib.pyplot` as `plt` and `seaborn` as `sns`. Create a histogram of `Total Spent` using `sns.histplot()`. What does the distribution look like?

b) Create a grouped bar plot showing total revenue (sum of `Total Spent`) per item for each day of the week. Use groupby() with both Day of Week and Item, then use `sns.barplot()` with `hue="Item"`. Does any product stand out on particular days?

c) Create a count plot showing how many transactions there are per item using `sns.countplot()`. Sort the bars from most to least frequent. **Hint:** use the `order` parameter with the output of `value_counts().index`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# YOUR CODE HERE


### Exercise 9: Visualizing relationships

a) Create a new column called `Day of Week` that contains the weekday name (Monday, Tuesday, ...) extracted from `Transaction Date`. Then visualize total sales (sum of `Total Spent`) by day of the week using a bar plot. Is there a trend across the week?

**Hint:** use `.dt.day_name()` to get the weekday name. To keep the days in calendar order rather than alphabetical, you can pass an explicit order to the plot.

b) Create a scatter plot with `Day of Week` on the x-axis and `Total Spent` on the y-axis, coloured by `Item` using the `hue` parameter. Does spending behaviour differ by product across the week?

c) Create a bar plot showing the total revenue (sum of `Total Spent`) per item. **Hint:** first compute the sum with `groupby()`, then use `.plot.bar()` or `sns.barplot()`.

d) If the `Transaction Date` column was successfully converted to datetime, try creating a line plot showing total daily revenue over time. **Hint:** group by date, sum `Total Spent`, and use `.plot()`.

In [ ]:
# YOUR CODE HERE
